# 🛠️ Step 2: Interactive Feature Engineering
Transforming clean data into a high-performance feature matrix for modeling. This involves scaling numerical features and encoding categorical ones.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import joblib
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

cleaned_path = '../data/processed/credit_risk_cleaned.csv'
df = pd.read_csv(cleaned_path)
print(f'Cleaned dataset shape: {df.shape}')
df.head()

## 1. Feature Selection & Pipeline Setup
Separating variables by type to apply consistent transformations.

In [ ]:
# Define features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']

# Group columns by type
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print(f'Categorical: {categorical_cols}')
print(f'Numerical: {numerical_cols}')

## 2. Transformation Pipeline (Scaling & Encoding)
Applying **StandardScaler** for numeric stability and **OneHotEncoder** for categorical interoperability.

### 🔍 Key Concepts in Feature Engineering

#### 1. One-Hot Encoding (OHE)
Categorical features like `loan_intent` (e.g., 'EDUCATION', 'MEDICAL') are converted into binary "dummy" columns. 
*   **Example**: If intent is 'MEDICAL', it becomes: `[Intent_Medical=1, Intent_Education=0, ...]`.
*   **Why?**: Models need numbers, but simple labeling (1, 2, 3) implies an incorrect ranking/order.

#### 2. Standard Scaling
Numerical features like `person_income` and `person_age` have vastly different ranges. 
*   **Method**: We shift the mean to **0** and scale the standard deviation to **1**.
*   **Example**: After scaling, an income of **0** represents the population **average**.
*   **Why?**: It prevents features with huge values (like Income) from drowning out features with small values (like Age).


In [ ]:
# Build the Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# Fit and Transform
X_processed = preprocessor.fit_transform(X)

# Extract new feature names
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names)

# Create final DataFrame
X_final = pd.DataFrame(X_processed, columns=all_feature_names)
print(f'New Feature Matrix Shape: {X_final.shape}')
X_final.head()

## 3. Visualizing Transformation Results
Checking how scaling affected the distribution of key features.

In [ ]:
# Compare before and after scaling for a feature (e.g., person_income)
fig = px.histogram(X_final, x='person_income', title='Distribution of Standardized Income', 
                   color_discrete_sequence=['#4C72B0'])
fig.show()

## 4. Save Features & Preprocessor Artifact
Saving the processed data for training and the preprocessor object for model serving (Inference).

### 💾 Why save the `preprocessor.joblib`?
The preprocessor is just as important as the model itself because it ensures **Consistency**:

1.  **Exact Scaling**: It remembers the *mean* and *standard deviation* of your training data. New users must be scaled using these same numbers to get accurate results.
2.  **Category Mapping**: It remembers that 'RENT' corresponds to a specific one-hot column index. Without it, your categories might get shuffled during production.
3.  **Production Readiness**: In a real app (API), you will load this file to "translate" raw user input into the format the model expects.


In [ ]:
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# Save data
X_final.to_csv(os.path.join(output_dir, 'features.csv'), index=False)
y.to_csv(os.path.join(output_dir, 'target.csv'), index=False)

# Save preprocessor artifact
joblib.dump(preprocessor, os.path.join(output_dir, 'preprocessor.joblib'))

print(f'Feature engineering artifact saved to: {output_dir}')